# 💻 Notebook do Aluno — Aula 06: Pipeline RAG completo

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 06/14 — Módulo 2: RAG · load → split → embed → retrieve → generate**  
**⏱️ 1h40min**  
**📄 PyMuPDF · RecursiveCharacterTextSplitter**  
**🔁 Andaime 50%**  

---

## 🎯 Objetivo da aula

Construir um pipeline RAG completo funcional em ~50 linhas. O LLM responde sobre documentos reais que nunca estiveram no treinamento — com citação de página e trecho. Essa é a fundação do CKP02.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime da aula.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

---

## 🧩 Andaime da aula — complete as lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-community langchain-ollama pymupdf chromadb langchain-text-splitters -q

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
import os
from google.colab import userdata, files

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
# 👉 LACUNA 1: faça upload e carregue os PDFs do domínio
uploaded  = files.upload()
pdf_paths = [___]  # lista de caminhos dos PDFs enviados
paginas   = []
for p in pdf_paths:
    paginas.extend(PyMuPDFLoader(___).load())

# 👉 LACUNA 2: crie o splitter e divida as páginas em chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=___,     # 800 é um bom ponto de partida
    chunk_overlap=___,  # 100 para não perder contexto nas bordas
)
chunks = splitter.split_documents(___)
print(f"Chunks: {len(chunks)}")

# 👉 LACUNA 3: crie o vector store com nomic-embed-text
embeddings = OllamaEmbeddings(model=___)
db = Chroma.from_documents(___, embedding=___, persist_directory="/content/ckp02")
retriever = db.as_retriever(search_kwargs={"k":3})

# 👉 LACUNA 4: monte e invoque a chain RAG com grounding e citação
chain_rag = (
    {"contexto": retriever | RunnableLambda(formatar_contexto),
     "pergunta": ___,
     "nome_doc": RunnableLambda(lambda _: "PDFs do domínio")}
    | prompt | ChatOllama(model="gpt-oss:120b", temperature=0) | StrOutputParser()
)
print(chain_rag.invoke(___))  # sua pergunta sobre o domínio

---

## ✍️ Suas anotações

Registre aqui suas observações sobre o andaime e os exercícios (qualidade dos resultados, comparações e conclusões).

---

## 🏋️ Exercícios da Aula 06

Quatro exercícios práticos sobre o pipeline RAG — as 5 etapas ponta a ponta, o grounding estrito no prompt, o carregamento de PDFs com PyMuPDF e a prova anti-alucinação.

Rode no Google Colab, na ordem, com os PDFs do domínio do grupo: as perguntas da prova devem ter respostas nos documentos — exceto a pergunta fora deles, que deve retornar o fallback.


### Exercício 1 — As 5 etapas do pipeline RAG · ★★☆ · 10 min

*Individual · Colab*

1. Complete os parâmetros do splitter: `chunk_size=___` e `chunk_overlap=___` — os valores usados na aula.
2. Complete o retriever com o `k` da aula em `search_kwargs=___`.
3. Preencha a pergunta do domínio e invoque a chain no `⑤ GENERATE`.
4. Anote em comentário o objeto que cada etapa produziu (Document, chunks, top-k, string).

> **💡 Dica:** `metadata["page"]` sobrevive ao splitter — é ele que torna a citação `(fonte, página X)` possível.


In [ ]:
# Exercício 1 — o pipeline inteiro, etapa a etapa (mini corpus)
from langchain_core.documents import Document

paginas_ex = [
    Document(page_content="Manual do produto. Garantia de 12 meses. O prazo de entrega é de 5 dias úteis após a confirmação do pagamento.", metadata={"source": "manual_demo.pdf", "page": 0}),
    Document(page_content="Suporte técnico pelo portal e pelo chat, em dias úteis, das 9h às 18h, com atendimento em até 48 horas úteis.", metadata={"source": "manual_demo.pdf", "page": 1}),
]
print(f"① LOAD → {len(paginas_ex)} Documents")

# 👉 LACUNA 1 e 2: os parâmetros do splitter — os valores usados na aula (800 e 100)
splitter = RecursiveCharacterTextSplitter(chunk_size=___, chunk_overlap=___)
chunks = splitter.split_documents(paginas_ex)
print(f"② SPLIT → {len(chunks)} chunks · metadata: {chunks[0].metadata}")

embeddings_ex = OllamaEmbeddings(model="nomic-embed-text")
db_ex = Chroma.from_documents(chunks, embeddings_ex, collection_name="ex01_pipeline")
print(f"③ EMBED + STORE → {db_ex._collection.count()} chunks")

# 👉 LACUNA 3: o retriever com k=3
retriever_ex = db_ex.as_retriever(search_kwargs=___)
print(f"④ RETRIEVE → {len(retriever_ex.invoke('qual é o prazo de entrega?'))} chunks para a pergunta")

def formatar_contexto_ex(docs) -> str:
    return "\n\n---\n\n".join(
        f"[{d.metadata.get('source','?')}, pág. {d.metadata.get('page',0)+1}]\n{d.page_content}"
        for d in docs
    )

PROMPT_RAG_EX = """<persona>
Você é um assistente especializado em responder perguntas
com base EXCLUSIVAMENTE nos documentos fornecidos.
</persona>

<instrucoes>
- Responda SOMENTE com informações do contexto.
- Sempre cite a fonte: (fonte: manual_demo.pdf, página X).
- Se a resposta não estiver no contexto, diga:
  "Não encontrei essa informação nos documentos fornecidos."
</instrucoes>

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta}
</pergunta>"""

chain_rag_ex = (
    {"contexto": retriever_ex | RunnableLambda(formatar_contexto_ex),
     "pergunta": RunnablePassthrough(),
     "nome_doc": RunnableLambda(lambda _: "manual_demo.pdf")}
    | ChatPromptTemplate.from_template(PROMPT_RAG_EX)
    | ChatOllama(model="gpt-oss:120b", temperature=0) | StrOutputParser()
)
print("⑤ GENERATE →")
# 👉 LACUNA 4: invoque a chain com uma pergunta real do domínio
print(chain_rag_ex.invoke(___))


### Exercício 2 — Prompt que perde o grounding · ★★☆ · 10 min

*Individual · Colab*

1. Rode o prompt "vazado" na chain e veja os riscos (alucinação, citação inventada).
2. Complete as 3 lacunas do template com grounding estrito: o que responde, a página citada e o fallback.
3. Invoque a chain com uma pergunta que NÃO está nos documentos.
4. Confira: a resposta deve ser o fallback "Não encontrei essa informação nos documentos fornecidos."

> **💡 Dica:** o grounding estrito é o que a RAGAS da Aula 07 medirá como `faithfulness`.


In [ ]:
# Exercício 2 — restaure o grounding estrito no prompt (usa a chain do Exercício 1)
PROMPT_VAZADO = """Você é um assistente prestativo.
Responda a pergunta usando o contexto e, se precisar,
complete com seu conhecimento geral.

<contexto>{contexto}</contexto>
<pergunta>{pergunta}</pergunta>"""
# Riscos da versão vazada: alucinação + citação de página inventada.

# 👉 LACUNA 1: complete as 3 lacunas do template com grounding estrito
PROMPT_RAG = """<persona>
Você é um assistente especializado em responder perguntas
com base EXCLUSIVAMENTE nos documentos fornecidos.
</persona>

<instrucoes>
- Responda SOMENTE com informações do ___.
- Sempre cite a fonte: (fonte: {nome_doc}, página ___).
- Se a resposta não estiver no contexto, diga:
  "___"
- Nunca invente, extrapole ou use conhecimento externo.
</instrucoes>

<contexto>
{contexto}
</contexto>

<pergunta>
{pergunta}
</pergunta>"""
prompt_corrigido = ChatPromptTemplate.from_template(PROMPT_RAG)

chain_prova = (
    {"contexto": retriever_ex | RunnableLambda(formatar_contexto_ex),
     "pergunta": RunnablePassthrough(),
     "nome_doc": RunnableLambda(lambda _: "manual_demo.pdf")}
    | prompt_corrigido | ChatOllama(model="gpt-oss:120b", temperature=0) | StrOutputParser()
)

# 👉 LACUNA 2: invoque a chain com uma pergunta que NÃO está nos documentos
print(chain_prova.invoke(___))
# Esperado: "Não encontrei essa informação nos documentos fornecidos."


### Exercício 3 — Carregamento de PDF com PyMuPDF · ★★☆ · 10 min

*Individual · Colab*

1. Envie um PDF do domínio pelo seletor e confirme o caminho em `pdf_path`.
2. Complete o loader com o caminho do arquivo.
3. Chame o método que carrega o arquivo — cada página vira um Document.
4. Complete a chave do `metadata` que guarda o número da página.

> **💡 Dica:** o contrato do loader é 1 página = 1 Document — e é o `metadata["page"]` que a citação do prompt usa.


In [ ]:
# Exercício 3 — carregar PDFs com PyMuPDFLoader
from google.colab import files
uploaded = files.upload()                      # abre o seletor de arquivos
pdf_path = list(uploaded.keys())[0]

# 👉 LACUNA 1: crie o loader com o caminho do PDF enviado
loader = PyMuPDFLoader(___)

# 👉 LACUNA 2: chame o método que carrega o arquivo — cada página vira um Document
paginas_pdf = loader.___
print(f"Páginas carregadas: {len(paginas_pdf)}")

# 👉 LACUNA 3: a chave do metadata que guarda o número da página
print(f"Metadados da pág. 1: {paginas_pdf[0].___}")
print(f"Trecho da pág. 1:\n{paginas_pdf[0].page_content[:300]}")


### Exercício 4 — Prova anti-alucinação · ★★☆ · 10 min

*Individual · Colab*

1. Liste 2 perguntas com resposta nos PDFs e 1 claramente fora deles.
2. Invoque a chain do Exercício 2 com cada pergunta.
3. Confira as citações: as duas primeiras devem citar (fonte, página X).
4. Confira a terceira: deve retornar o fallback, não inventar.

> **💡 Dica:** se a pergunta fora dos documentos alucinar, confirme `temperature=0` e reforce a instrução "Nunca invente, extrapole ou use conhecimento externo."


In [ ]:
# Exercício 4 — a prova anti-alucinação (usa a chain do Exercício 2)
perguntas_prova = [
    ___,   # 👉 LACUNA 1: pergunta COM resposta nos PDFs
    ___,   # 👉 LACUNA 2: outra pergunta COM resposta
    ___,   # 👉 LACUNA 3: pergunta SEM resposta nos PDFs
]
for q in perguntas_prova:
    print(f"\n📌 {q}")
    # 👉 LACUNA 4: invoque a chain com a pergunta
    print(___)
# Esperado: as 2 primeiras com citação (fonte, página X);
# a 3ª retorna "Não encontrei essa informação nos documentos fornecidos."
# Se a 3ª alucinar: confirme temperature=0 e reforce a instrução no prompt.


## 📚 Referências da aula

- Paper Lewis, P. et al. — "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks." NeurIPS, 2020. O paper original que cunhou o termo RAG. arxiv.org/abs/2005.11401
- Docs LangChain — RAG tutorial completo com PyMuPDF, Chroma e LCEL. python.langchain.com/docs/tutorials/rag
- Docs PyMuPDF — Documentação do loader LangChain com PyMuPDF. python.langchain.com/docs/integrations/document_loaders/pymupdf
- Docs RecursiveCharacterTextSplitter — Estratégias de chunking, parâmetros e separadores. python.langchain.com/docs/how_to/recursive_text_splitter
- Livro Goodfellow, I.; Bengio, Y.; Courville, A. — Deep Learning. Pearson, 2017. Cap. 15 — Representações distribuídas: a base teórica dos embeddings usados no RAG.

---

**→ Próxima Aula — Aula 07 · 21/09** — RAG avançado — chunking estratégico, reranking e RAGAS
  
Medir faithfulness e answer relevancy. Otimizar o pipeline. Entregar CKP02.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*